⚽ Proxecto Power BI: Análise de Datos de Fútbol

🎯 Obxectivo

Analizar estatísticas de equipos e xogadores de fútbol, cruzando datos de distintas fontes para obter información valiosa sobre rendemento, condicións meteorolóxicas durante os partidos e outros factores relevantes.

📁 Fontes de Datos

    1. Excel: Estatísticas de equipos (partidos xogados, vitorias, empates, derrotas, goles a favor e en contra).

    2. JSON: Datos meteorolóxicos durante os partidos, obtidos mediante scraping.

    3. Script de Python: Para realizar o scraping dos datos meteorolóxicos e procesalos.

    4. Spark-HDFS: Datos históricos de partidos, almacenados en formato Parquet.

🐍 Script de Scraping en Python

Utilizaremos requests e BeautifulSoup para obter datos meteorolóxicos de partidos desde unha fonte como Time and Date.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

# Lista de cidades e as súas rutas na URL
cities = {
    "Madrid": "madrid",
    "Barcelona": "barcelona",
    "Valencia": "valencia",
    "Sevilla": "seville",
    "Bilbao": "bilbao",
    "Zaragoza": "zaragoza",
    "Málaga": "malaga",
    "A Coruña": "la-coruna"
}

base_url = "https://www.timeanddate.com/weather/spain/"
cities_data = []

for city_name, city_path in cities.items():
    url = f"{base_url}{city_path}"
    print(f"Lendo {city_name}...")
    
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        temp_tag = soup.select_one(".h2")
        if temp_tag:
            temp_text = temp_tag.text.strip().replace("°C", "").split()[0]
            try:
                temperature = float(temp_text)
            except ValueError:
                temperature = None
        else:
            temperature = None

        cities_data.append({
            "City": city_name,
            "Temperature": temperature
        })

        time.sleep(1)  # Evitar sobrecarga do servidor
    except Exception as e:
        print(f"Erro lendo {city_name}: {e}")

# Gardar os datos
df = pd.DataFrame(cities_data)
df.to_csv("weather_spain.csv", index=False)
with open("weather_spain.json", "w", encoding="utf-8") as f:
    json.dump(cities_data, f, ensure_ascii=False, indent=2)

print("\n✅ Datos gardados correctamente!")
print(df)


Lendo Madrid...
Lendo Barcelona...
Lendo Valencia...
Lendo Sevilla...
Lendo Bilbao...
Lendo Zaragoza...
Lendo Málaga...
Lendo A Coruña...

✅ Datos gardados correctamente!
        City  Temperature
0     Madrid         24.0
1  Barcelona         19.0
2   Valencia         22.0
3    Sevilla         27.0
4     Bilbao         23.0
5   Zaragoza         26.0
6     Málaga         25.0
7   A Coruña         17.0


Este script recolle as temperaturas das principais cidades de España e gárdaas en formato JSON e CSV para a súa posterior análise.

🐼 Procesamento de Datos con Pandas en Power BI

In [3]:
import pandas as pd
import json

# Ler JSON exportado do scraping
with open("weather_spain.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Limpeza básica
df = df.dropna()
df["City"] = df["City"].str.strip().str.title()

df

,City,Temperature
0,Madrid,24.0
1,Barcelona,19.0
2,Valencia,22.0
3,Sevilla,27.0
4,Bilbao,23.0
5,Zaragoza,26.0
6,Málaga,25.0
7,A Coruña,17.0


Este script limpa os datos meteorolóxicos e prepáraos para a súa integración con outras fontes de datos en Power BI.

🔗 Integración con Spark-HDFS

Para integrar datos almacenados en HDFS, podemos utilizar PySpark:

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FootballData").getOrCreate()

# Ler datos de partidos desde HDFS
df = spark.read.parquet("hdfs://localhost:9000/datos_futbol/partidos.parquet")
df.show()


ModuleNotFoundError: No module named 'pyspark'

Estes datos poden incluír información detallada sobre partidos, como resultados, estatísticas de xogadores, etc.

In [ ]:
# Crear dataset de exemplo para asistencia con más equipos
asistencia_data = [
    {"Equipo": "Real Madrid", "Estadio": "Santiago Bernabéu", "Cidade": "Madrid", "Asistencia": 70000, "Capacidade": 81044},
    {"Equipo": "FC Barcelona", "Estadio": "Estadi Olímpic", "Cidade": "Barcelona", "Asistencia": 50000, "Capacidade": 55000},
    {"Equipo": "Sevilla FC", "Estadio": "Ramón Sánchez-Pizjuán", "Cidade": "Sevilla", "Asistencia": 35000, "Capacidade": 43883},
    {"Equipo": "Athletic Club", "Estadio": "San Mamés", "Cidade": "Bilbao", "Asistencia": 42000, "Capacidade": 53289},
    {"Equipo": "Atlético Madrid", "Estadio": "Wanda Metropolitano", "Cidade": "Madrid", "Asistencia": 68000, "Capacidade": 68000},
    {"Equipo": "Real Betis", "Estadio": "Benito Villamarín", "Cidade": "Sevilla", "Asistencia": 50000, "Capacidade": 60000},
    {"Equipo": "Valencia CF", "Estadio": "Mestalla", "Cidade": "Valencia", "Asistencia": 46000, "Capacidade": 55000},
    {"Equipo": "Villarreal CF", "Estadio": "Estadio de la Cerámica", "Cidade": "Villarreal", "Asistencia": 24000, "Capacidade": 25000},
    {"Equipo": "Granada CF", "Estadio": "Nuevo Los Cármenes", "Cidade": "Granada", "Asistencia": 22000, "Capacidade": 22500},
    {"Equipo": "Celta de Vigo", "Estadio": "Abanca-Balaídos", "Cidade": "Vigo", "Asistencia": 29000, "Capacidade": 30000},
    {"Equipo": "Real Sociedad", "Estadio": "Anoeta", "Cidade": "San Sebastián", "Asistencia": 32000, "Capacidade": 39000},
    {"Equipo": "Getafe CF", "Estadio": "Coliseum Alfonso Pérez", "Cidade": "Getafe", "Asistencia": 17000, "Capacidade": 17000},
    {"Equipo": "Espanyol", "Estadio": "RCDE Stadium", "Cidade": "Barcelona", "Asistencia": 29000, "Capacidade": 40000},
    {"Equipo": "Alavés", "Estadio": "Mendizorroza", "Cidade": "Vitoria", "Asistencia": 19000, "Capacidade": 19800},
    {"Equipo": "Osasuna", "Estadio": "El Sadar", "Cidade": "Pamplona", "Asistencia": 23000, "Capacidade": 23600},
]
asistencia_df = pd.DataFrame(asistencia_data)
asistencia_df.to_csv("asistencia_estadios.csv", index=False)

asistencia_df


,Equipo,Estadio,Cidade,Asistencia,Capacidade
0,Real Madrid,Santiago Bernabéu,Madrid,70000,81044
1,FC Barcelona,Estadi Olímpic,Barcelona,50000,55000
2,Sevilla FC,Ramón Sánchez-Pizjuán,Sevilla,35000,43883
3,Athletic Club,San Mamés,Bilbao,42000,53289
4,Atlético Madrid,Wanda Metropolitano,Madrid,68000,68000
5,Real Betis,Benito Villamarín,Sevilla,50000,60000
6,Valencia CF,Mestalla,Valencia,46000,55000
7,Villarreal CF,Estadio de la Cerámica,Villarreal,24000,25000
8,Granada CF,Nuevo Los Cármenes,Granada,22000,22500
9,Celta de Vigo,Abanca-Balaídos,Vigo,29000,30000


In [ ]:
import pandas as pd

# Crear dataset de exemplo para clima
clima_data = [
    {"Cidade": "Madrid", "Data": "2025-04-01", "Temperatura": 18.0, "Condición": "Despexado"},
    {"Cidade": "Barcelona", "Data": "2025-04-01", "Temperatura": 19.0, "Condición": "Nubes"},
    {"Cidade": "Sevilla", "Data": "2025-04-01", "Temperatura": 23.0, "Condición": "Sol"},
    {"Cidade": "Bilbao", "Data": "2025-04-01", "Temperatura": 15.0, "Condición": "Choiva"},
    {"Cidade": "Valencia", "Data": "2025-04-01", "Temperatura": 20.0, "Condición": "Nubes"},
    {"Cidade": "Vigo", "Data": "2025-04-01", "Temperatura": 16.5, "Condición": "Sol"},
    {"Cidade": "Granada", "Data": "2025-04-01", "Temperatura": 22.0, "Condición": "Despexado"},
    {"Cidade": "San Sebastián", "Data": "2025-04-01", "Temperatura": 17.0, "Condición": "Nubes"},
]
clima_df = pd.DataFrame(clima_data)
clima_df["Data"] = pd.to_datetime(clima_df["Data"])
clima_df.to_csv("clima_partidos.csv", index=False)

# Ver los primeros registros
clima_df.head()


,Cidade,Data,Temperatura,Condición
0,Madrid,2025-04-01,18.0,Despexado
1,Barcelona,2025-04-01,19.0,Nubes
2,Sevilla,2025-04-01,23.0,Sol
3,Bilbao,2025-04-01,15.0,Choiva
4,Valencia,2025-04-01,20.0,Nubes
